# Loading Data with HLA or MHC information

TCR repertoire is understand in the context of the HLA/MHC background in which it resides. The antigenic specificty of a TCR in a host is not only determined by the structure of the TCR but also by what HLA/MHC molecules are present as these dictate the peptides that can be presented and recognized by a particular TCR repertoire.

When comparing the TCR repertoires between inbred mice, since they share the same MHC background, their TCR repertoires are directly comparable. A TCR seen in any animal will be recognizing the same epitope.

However, when comparing human samples with HLA heterogeneity, the same TCR in two humans does not mean they are recognizing the same epitopes as what is presented is determined by the HLA of the human. Therefore, one wants to make comparisons of repertoire in context of the HLA background the TCR was seen in.

In this tutorial, we will demonstrate how to load HLA information along with TCR-Seq data.

## Preparing Data

We've created a toy dataset under Data called 'Human_HLA_Tutorial' to walk through how to incorporate HLA into your TCR-Seq analysis. In that folder, you should see a folder called 'Data' with 10 samples labeled 'Sample_N.tsv'. Additionally, in that same folder, you will see a file called HLA.csv where the first column lists the file names where the TCR-Seq data is stored along with 6 columns for HLA information. It is important that the HLA information is provided in this format in a csv formatted file.

These HLA values can be any categorical value the user wants and HLA can be encoded in anyway the user wants as long as it's consistent across all samples. DeepTCR takes the number of unique HLA types in this file and creates categories from them for the purpose of encoding this information into the model.

As demonstrated, it is simple to load the TCR and HLA information once it has been prepared properly. Of note, if any TCR-Seq files do not have corresponding HLA information as provided in the HLA.csv file, those samples will be dropped from the analysis.

We can also visualize here how many HLA categories were parsed from the data.

While the HLA loci are very genetically diverse, the idea of HLA supertypes is that there are biologically functional groupings of HLA based on how they bind antigen. In DeepTCR, a user can choose to transform the HLA information from allele (i.e. A0101) to supertype (i.e. A01). In order to do this, it is a simple as setting the use_hla_supertype to True. For this method to work, HLA MUST be provided in the following format -  A0101. Formats such as A*0101 or HLA-A*0101 will not be recognized.

As you can see, the C alleles were removed as there are no well understood groupings as of yet (if this changes, we will update this in later versions of DeepTCR). However, if one wants to include HLA alleles that do not fall into group as specified by the publication above, one can set another parameter (keep_non_supertype_alleles) to True.

In [13]:
# --- Cell 1: Setup & Paths (no hardcoded absolute paths) ---
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd

# 1) Find the repo root (directory that contains both 'Data' and 'tutorials')
def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "Data").exists() and (p / "tutorials").exists():
            return p
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
print(f"Repo root: {REPO_ROOT}")

# 2) Ensure we can import DeepTCR from a local clone
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
try:
    from DeepTCR.DeepTCR import DeepTCR_U
    print("DeepTCR import: OK")
except Exception as e:
    print("DeepTCR import failed. sys.path is:\n", sys.path)
    raise

# 3) Paths for this tutorial (relative to repo root)
TUTORIAL_NAME = "HLA Tutorial"
HLA_TUT_ROOT = REPO_ROOT / "Data" / "Human_HLA_Tutorial"
DATA_RAW      = HLA_TUT_ROOT / "Data"            # original TSVs
DATA_PATCHED  = HLA_TUT_ROOT / "Data_patched"    # we will create if needed
HLA_CSV_CLEAN = HLA_TUT_ROOT / "HLA_clean.csv"
HLA_CSV_RAW   = HLA_TUT_ROOT / "HLA.csv"
HLA_CSV       = HLA_CSV_CLEAN if HLA_CSV_CLEAN.exists() else HLA_CSV_RAW

print("\n--- Paths ---")
print("DATA_RAW     :", DATA_RAW)
print("DATA_PATCHED :", DATA_PATCHED)
print("HLA_CSV      :", HLA_CSV)

# 4) Quick sanity checks on inputs
def list_tsvs(d: Path):
    return sorted([p for p in d.glob("*.tsv")])

raw_tsvs = list_tsvs(DATA_RAW)
print("\n--- Raw TSV files ---")
print(f"count = {len(raw_tsvs)}")
print([p.name for p in raw_tsvs[:10]])

if not raw_tsvs:
    print("\n⚠️  No TSV files found in DATA_RAW. "
          "Confirm the repo structure: Data/Human_HLA_Tutorial/Data/*.tsv")

if not HLA_CSV.exists():
    raise FileNotFoundError(f"HLA file not found at {HLA_CSV}")

# 5) Notebook knobs (you can change these later if you like)
PATCH_MISSING_GENES = True   # fills blank/NaN V/D/J gene names -> 'Unknown' into Data_patched
USE_HLA_SUPERTYPE   = False  # if True, map HLA alleles to supertypes (optional, later cell)

print("\n--- Options ---")
print("PATCH_MISSING_GENES:", PATCH_MISSING_GENES)
print("USE_HLA_SUPERTYPE  :", USE_HLA_SUPERTYPE)

# 6) Show a peek at the HLA CSV (top 5 rows)
print("\n--- HLA CSV head ---")
try:
    display(pd.read_csv(HLA_CSV).head())
except Exception as e:
    print(f"(Could not preview HLA CSV: {e})")


Repo root: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master
DeepTCR import: OK

--- Paths ---
DATA_RAW     : /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/Data
DATA_PATCHED : /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/Data_patched
HLA_CSV      : /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/HLA_clean.csv

--- Raw TSV files ---
count = 10
['Sample_1.tsv', 'Sample_10.tsv', 'Sample_2.tsv', 'Sample_3.tsv', 'Sample_4.tsv', 'Sample_5.tsv', 'Sample_6.tsv', 'Sample_7.tsv', 'Sample_8.tsv', 'Sample_9.tsv']

--- Options ---
PATCH_MISSING_GENES: True
USE_HLA_SUPERTYPE  : False

--- HLA CSV head ---


,__sample__,File,0,1,2,3,4,5
0,SAMPLE_1,Sample_1.tsv,A1101,A2501,B1801,B5201,C1202,C1203
1,SAMPLE_2,Sample_2.tsv,A1101,A2501,B1801,B5201,C1202,C1203
2,SAMPLE_3,Sample_3.tsv,A1101,A2501,B1801,B5201,C1202,C1203
3,SAMPLE_4,Sample_4.tsv,A1101,A2501,B1801,B5201,C1202,C1203
4,SAMPLE_5,Sample_5.tsv,A0201,A2902,B3801,B4501,C0602,C1203


In [14]:
# --- Cell 2: Patch missing V/D/J gene names into Data_patched/ ---

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATCHED.mkdir(exist_ok=True)

TARGET_COLS = {
    "vGeneName": "vGeneName",
    "dGeneName": "dGeneName",
    "jGeneName": "jGeneName",
    "aminoAcid": "aminoAcid",
}

def _to_unknown(x):
    # Normalize "missing-ish" things to 'Unknown'
    if x is None:
        return "Unknown"
    if isinstance(x, float) and np.isnan(x):
        return "Unknown"
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return "Unknown"
    return s

patched_report = []
for tsv in sorted(DATA_RAW.glob("*.tsv")):
    df = pd.read_csv(tsv, sep="\t")
    before = {}
    after = {}

    # Count nulls before
    for col in TARGET_COLS.values():
        if col in df.columns:
            before[col] = int(df[col].isna().sum())
        else:
            before[col] = None  # column not present

    # Ensure columns exist (if any are totally missing, we create them as Unknown)
    for col in ["vGeneName", "dGeneName", "jGeneName"]:
        if col not in df.columns:
            df[col] = "Unknown"

    # Fill + normalize V/D/J + aminoAcid
    for col in ["vGeneName", "dGeneName", "jGeneName", "aminoAcid"]:
        if col in df.columns:
            df[col] = df[col].map(_to_unknown)

    # Drop rows with missing/empty CDR3 aa after normalization
    if "aminoAcid" in df.columns:
        df = df[df["aminoAcid"] != "Unknown"].copy()

    # Count nulls after (should be 0)
    for col in ["vGeneName", "dGeneName", "jGeneName"]:
        after[col] = int(df[col].isna().sum())

    # Write patched file
    out = DATA_PATCHED / tsv.name
    df.to_csv(out, sep="\t", index=False)

    patched_report.append({
        "file": tsv.name,
        "rows_out": len(df),
        "v_null_before": before.get("vGeneName"),
        "d_null_before": before.get("dGeneName"),
        "j_null_before": before.get("jGeneName"),
        "v_null_after": after.get("vGeneName"),
        "d_null_after": after.get("dGeneName"),
        "j_null_after": after.get("jGeneName"),
    })

rep = pd.DataFrame(patched_report)
print("Patched files written to:", DATA_PATCHED)
display(rep)

# Quick peek at one patched file
example_patched = sorted(DATA_PATCHED.glob("*.tsv"))[0]
print("\nExample patched file:", example_patched.name)
display(pd.read_csv(example_patched, sep="\t").head(5)[["aminoAcid","vGeneName","dGeneName","jGeneName"]])


Patched files written to: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/Data_patched


,file,rows_out,v_null_before,d_null_before,j_null_before,v_null_after,d_null_after,j_null_after
0,Sample_1.tsv,312,119,75,2,0,0,0
1,Sample_10.tsv,802,311,222,2,0,0,0
2,Sample_2.tsv,111,34,36,0,0,0,0
3,Sample_3.tsv,188,91,59,0,0,0,0
4,Sample_4.tsv,525,199,127,5,0,0,0
5,Sample_5.tsv,213,81,50,2,0,0,0
6,Sample_6.tsv,81,44,19,0,0,0,0
7,Sample_7.tsv,252,113,53,1,0,0,0
8,Sample_8.tsv,220,80,66,0,0,0,0
9,Sample_9.tsv,426,172,119,1,0,0,0



Example patched file: Sample_1.tsv


,aminoAcid,vGeneName,dGeneName,jGeneName
0,CAISTGDSNQPQHF,TCRBV06-08,TCRBD01-01,TCRBJ01-05
1,CASSLFQLSNTGELFF,TCRBV28-01,TCRBD01-01,TCRBJ02-02
2,CSARGGPDTQYF,TCRBV29-01,TCRBD02-01,TCRBJ02-03
3,CASSQESFDVEGAKTEAFF,Unknown,TCRBD01-01,TCRBJ01-01
4,CASSSDYGQSYSYEQYF,TCRBV06-08,TCRBD01-01,TCRBJ02-07


In [19]:
# Cell: rebuild Data_patched WITH a numeric 'count' column, then preview

from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path("/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master")
DATA_RAW = REPO / "Data/Human_HLA_Tutorial/Data"
DATA_PATCHED = REPO / "Data/Human_HLA_Tutorial/Data_patched"

DATA_PATCHED.mkdir(parents=True, exist_ok=True)

def pick_col(df, candidates):
    # pick a column by trying several candidate names, case-insensitive, trimmed
    norm = {c.strip().lower(): c for c in df.columns}
    for want in candidates:
        if want.lower() in norm:
            return norm[want.lower()]
    # also try loose contains matching
    for k, orig in norm.items():
        for want in candidates:
            if want.lower() in k:
                return orig
    raise KeyError(f"None of {candidates} found. Available: {list(df.columns)}")

summary = []

for tsv in sorted(DATA_RAW.glob("*.tsv")):
    df = pd.read_csv(tsv, sep="\t", dtype=str)  # read as strings first
    # locate required columns in the raw files
    col_aa = pick_col(df, ["aminoAcid"])
    col_cnt = pick_col(df, ["count (templates/reads)", "count", "templates", "reads"])
    col_v  = pick_col(df, ["vGeneName", "vGene"])
    col_d  = pick_col(df, ["dGeneName", "dGene"])
    col_j  = pick_col(df, ["jGeneName", "jGene"])

    # build new frame with required schema
    out = pd.DataFrame({
        "aminoAcid": df[col_aa].astype(str).str.strip(),
        "count":     pd.to_numeric(df[col_cnt], errors="coerce"),
        "vGeneName": df[col_v].astype(str).str.strip(),
        "dGeneName": df[col_d].astype(str).str.strip(),
        "jGeneName": df[col_j].astype(str).str.strip(),
    })

    # fill invalid/missing counts with 1
    n_cnt_null = int(out["count"].isna().sum())
    out["count"] = out["count"].fillna(1).astype(int)

    # replace missing/empty genes with 'Unknown'
    for g in ["vGeneName", "dGeneName", "jGeneName"]:
        out[g] = out[g].replace({"nan": np.nan, "None": np.nan, "": np.nan})
        out[g] = out[g].fillna("Unknown")

    # drop sequences with missing amino acids
    before = len(out)
    out = out[out["aminoAcid"].notna() & (out["aminoAcid"].str.len() > 0)]
    dropped_empty_aa = before - len(out)

    # write patched file (tab-separated)
    out_path = DATA_PATCHED / tsv.name
    out.to_csv(out_path, sep="\t", index=False)

    summary.append((tsv.name, len(out), n_cnt_null, dropped_empty_aa))

# show a compact summary and preview first file
print(f"Patched {len(summary)} files into: {DATA_PATCHED}")
print("file\trows_out\tcount_null_filled\temptyAA_dropped")
for row in summary:
    print(f"{row[0]}\t{row[1]}\t{row[2]}\t{row[3]}")

# quick preview of one patched file
example = sorted(DATA_PATCHED.glob('*.tsv'))[0]
print("\nExample patched file:", example.name)
print(pd.read_csv(example, sep="\t").head())


Patched 10 files into: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/Data_patched
file	rows_out	count_null_filled	emptyAA_dropped
Sample_1.tsv	370	0	0
Sample_10.tsv	972	0	0
Sample_2.tsv	126	0	0
Sample_3.tsv	244	0	0
Sample_4.tsv	634	0	0
Sample_5.tsv	247	0	0
Sample_6.tsv	101	0	0
Sample_7.tsv	311	0	0
Sample_8.tsv	266	0	0
Sample_9.tsv	514	0	0

Example patched file: Sample_1.tsv
             aminoAcid  count   vGeneName   dGeneName   jGeneName
0       CAISTGDSNQPQHF      7  TCRBV06-08  TCRBD01-01  TCRBJ01-05
1     CASSLFQLSNTGELFF      4  TCRBV28-01  TCRBD01-01  TCRBJ02-02
2         CSARGGPDTQYF      3  TCRBV29-01  TCRBD02-01  TCRBJ02-03
3  CASSQESFDVEGAKTEAFF      2     Unknown  TCRBD01-01  TCRBJ01-01
4    CASSSDYGQSYSYEQYF      2  TCRBV06-08  TCRBD01-01  TCRBJ02-07


In [20]:
# Cell 2: Load patched data into DeepTCR_U and verify
from pathlib import Path
import numpy as np
import pandas as pd
from DeepTCR.DeepTCR import DeepTCR_U

REPO = Path("/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master")
DATA_PATCHED = REPO / "Data/Human_HLA_Tutorial/Data_patched"
HLA_MATCH = REPO / "Data/Human_HLA_Tutorial/HLA_matched.csv"   # created earlier; keeps only rows that exist on disk

print("DeepTCR from:", (REPO / "DeepTCR").resolve())
print("Patched TSV dir:", DATA_PATCHED)
print("HLA CSV:", HLA_MATCH)

# Instantiate a fresh workspace so we don't reuse any prior broken caches
DTCRU = DeepTCR_U('HLA Tutorial')  # writes to tutorials/HLA Tutorial/

# Load with CORRECT indices for the patched schema:
#   0: aminoAcid, 1: count, 2: vGeneName, 3: dGeneName, 4: jGeneName
print("\nLoading Data...")
DTCRU.Get_Data(
    directory=str(DATA_PATCHED),
    Load_Prev_Data=False,
    aa_column_beta=0,
    count_column=1,
    v_beta_column=2,
    d_beta_column=3,
    j_beta_column=4,
    hla=str(HLA_MATCH),
    use_hla_supertype=False,       # set True if you want supertypes instead of raw alleles
    keep_non_supertype_alleles=True
)
print("Data Loaded")

# ---- Sanity report (robust to attribute names) ----
n_sequences = None
n_samples_total = None
n_samples_unique = None

# sample_id is per-sequence; its unique values are sample files
if hasattr(DTCRU, "sample_id"):
    n_sequences = len(DTCRU.sample_id)
    n_samples_unique = len(np.unique(DTCRU.sample_id))

# try a few common names for the amino acid list
for cand in ["seqs_aa", "beta_sequences", "seqs_beta", "AA"]:
    if hasattr(DTCRU, cand) and n_sequences is None:
        try:
            n_sequences = len(getattr(DTCRU, cand))
        except Exception:
            pass

# classes label encoder (HLA bins) if present
classes = []
if hasattr(DTCRU, "lb"):
    try:
        classes = list(getattr(DTCRU, "lb").classes_)
    except Exception:
        pass

print("\nLoaded summary")
print("  rows (sequences):   ", n_sequences)
print("  samples (unique):   ", n_samples_unique)

# Cross-check against HLA CSV
hla_df = pd.read_csv(HLA_MATCH)
hla_df.columns = [str(c).strip() for c in hla_df.columns]
if "File" in hla_df.columns:
    files_in_hla = set(hla_df["File"].astype(str))
else:
    files_in_hla = set()

print("  HLA rows (csv):     ", len(hla_df))
print("  Example HLA rows:\n", hla_df.head(3), "\n")

# Quick count of distinct alleles seen (across columns 0..5)
allele_cols = [c for c in hla_df.columns if c.isdigit()]
alleles = pd.unique(pd.concat([hla_df[c] for c in allele_cols], ignore_index=True))
alleles = [a for a in alleles if pd.notna(a)]
print("  Distinct HLA alleles in CSV:", len(alleles))


DeepTCR from: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/DeepTCR
Patched TSV dir: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/Data_patched
HLA CSV: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/HLA_matched.csv

Loading Data...
Loading Data...


/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Embedding Sequences...
Data Loaded
Data Loaded

Loaded summary
  rows (sequences):    0
  samples (unique):    0
  HLA rows (csv):      10
  Example HLA rows:
   __sample__          File      0      1      2      3      4      5
0   SAMPLE_1  Sample_1.tsv  A1101  A2501  B1801  B5201  C1202  C1203
1   SAMPLE_2  Sample_2.tsv  A1101  A2501  B1801  B5201  C1202  C1203
2   SAMPLE_3  Sample_3.tsv  A1101  A2501  B1801  B5201  C1202  C1203 

  Distinct HLA alleles in CSV: 16


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
from DeepTCR.DeepTCR import DeepTCR_U

REPO = Path("/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master")
DATA_RAW  = REPO / "Data/Human_HLA_Tutorial/Data"          # original, wide TSVs
DATA_FIXED = REPO / "Data/Human_HLA_Tutorial/Data_fixed"   # new, same schema as original
HLA_CSV   = REPO / "Data/Human_HLA_Tutorial/HLA_matched.csv"  # you already created this (10/10 matched)

DATA_FIXED.mkdir(exist_ok=True)

# Column names used by the original HLA tutorial (wide TSVs)
COL_AA = "aminoAcid"
COL_COUNT = "count (templates/reads)"
COL_V = "vGeneName"
COL_D = "dGeneName"
COL_J = "jGeneName"

def patch_wide_tsv(in_path: Path, out_path: Path):
    df = pd.read_csv(in_path, sep="\t")
    # Ensure required columns are present
    for col in [COL_AA, COL_COUNT, COL_V, COL_D, COL_J]:
        if col not in df.columns:
            raise ValueError(f"{in_path.name}: missing required column: {col}")

    # Drop rows with empty AA
    df[COL_AA] = df[COL_AA].astype(str).str.strip()
    before = len(df)
    df = df[df[COL_AA] != ""].copy()

    # Fill missing V/D/J with 'Unknown'
    for col in [COL_V, COL_D, COL_J]:
        df[col] = df[col].where(df[col].notna() & (df[col].astype(str).str.strip() != ""), "Unknown")

    # Coerce count to numeric (keep original column name/index)
    df[COL_COUNT] = pd.to_numeric(df[COL_COUNT], errors="coerce").fillna(0).astype(int)

    df.to_csv(out_path, sep="\t", index=False)
    return before, len(df)

# Patch every file in the original directory (but keep ALL columns)
stats = []
for tsv in sorted(DATA_RAW.glob("*.tsv")):
    before, after = patch_wide_tsv(tsv, DATA_FIXED / tsv.name)
    stats.append((tsv.name, before, after))

print(f"Patched {len(stats)} files into: {DATA_FIXED}")
print("file\tbefore_rows\tafter_rows")
for s in stats:
    print(f"{s[0]}\t{s[1]}\t{s[2]}")

# ===== Load with DeepTCR using the ORIGINAL indices =====
# Original notebook indices (0-based): AA=1, COUNT=2, V=7, D=14, J=21
print("\nLoading (original schema + indices)...")
DTCRU = DeepTCR_U("HLA Tutorial (fixed-schema)")
DTCRU.Get_Data(
    directory=str(DATA_FIXED),
    Load_Prev_Data=False,
    aa_column_beta=1,
    count_column=2,
    v_beta_column=7,
    d_beta_column=14,
    j_beta_column=21,
    hla=str(HLA_CSV),
    use_hla_supertype=False,
    keep_non_supertype_alleles=True,
)

print("Data Loaded")

# ---- Sanity report ----
n_seq = getattr(DTCRU, "num_seq", None)
if n_seq is None and hasattr(DTCRU, "sample_id"):
    n_seq = len(DTCRU.sample_id)

print("\nLoaded summary")
print("  rows (sequences):   ", n_seq)
if hasattr(DTCRU, "sample_id"):
    print("  samples (unique):   ", len(np.unique(DTCRU.sample_id)))
if hasattr(DTCRU, "lb"):
    try:
        print("  HLA classes:        ", list(DTCRU.lb.classes_))
    except Exception:
        pass


Patched 10 files into: /Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/Data/Human_HLA_Tutorial/Data_fixed
file	before_rows	after_rows
Sample_1.tsv	370	370
Sample_10.tsv	972	972
Sample_2.tsv	126	126
Sample_3.tsv	244	244
Sample_4.tsv	634	634
Sample_5.tsv	247	247
Sample_6.tsv	101	101
Sample_7.tsv	311	311
Sample_8.tsv	266	266
Sample_9.tsv	514	514

Loading (original schema + indices)...
Loading Data...


/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Embedding Sequences...
Data Loaded
Data Loaded

Loaded summary
  rows (sequences):    0
  samples (unique):    0
  HLA classes:         ['None']


In [22]:
from pathlib import Path
import shutil, numpy as np
from DeepTCR.DeepTCR import DeepTCR_U

REPO = Path("/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master")
DATA_FIXED = REPO / "Data/Human_HLA_Tutorial/Data_fixed"     # the wide TSVs we just patched
ALL_DIR    = DATA_FIXED / "All"
HLA_CSV    = REPO / "Data/Human_HLA_Tutorial/HLA_matched.csv"

# 1) Ensure a class subfolder exists and copy files there (idempotent)
ALL_DIR.mkdir(exist_ok=True)
tsvs = sorted([p for p in DATA_FIXED.glob("*.tsv") if p.is_file()])
for p in tsvs:
    dst = ALL_DIR / p.name
    if not dst.exists():
        shutil.copy2(p, dst)

print("Class folders present under Data_fixed:", [d.name for d in DATA_FIXED.iterdir() if d.is_dir()])
print("Files in All/:", len(list(ALL_DIR.glob("*.tsv"))))

# 2) Load using ORIGINAL indices + classes=['All'] + HLA mapping
print("\nLoading via DeepTCR...")
DTCRU = DeepTCR_U("HLA Tutorial (All)")
DTCRU.Get_Data(
    directory=str(DATA_FIXED),
    classes=['All'],                 # <-- this is the key bit
    Load_Prev_Data=False,
    aa_column_beta=1,                # 0-based: aminoAcid
    count_column=2,                  # 0-based: count (templates/reads)
    v_beta_column=7,                 # 0-based: vGeneName
    d_beta_column=14,                # 0-based: dGeneName
    j_beta_column=21,                # 0-based: jGeneName
    hla=str(HLA_CSV),                # your matched HLA file
    use_hla_supertype=False,
    keep_non_supertype_alleles=True,
)

print("Data Loaded")

# 3) Quick sanity summary
n_seq = getattr(DTCRU, "num_seq", None)
if n_seq is None and hasattr(DTCRU, "sample_id"):
    n_seq = len(DTCRU.sample_id)

print("\nLoaded summary")
print("  rows (sequences):   ", n_seq)
if hasattr(DTCRU, "sample_id"):
    print("  samples (unique):   ", len(np.unique(DTCRU.sample_id)))
if hasattr(DTCRU, "lb"):
    try:
        print("  label classes:      ", list(DTCRU.lb.classes_))  # should be ['All']
    except Exception:
        pass


Class folders present under Data_fixed: ['All']
Files in All/: 10

Loading via DeepTCR...
Loading Data...


/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Embedding Sequences...
Data Loaded
Data Loaded

Loaded summary
  rows (sequences):    0
  samples (unique):    0
  label classes:       ['All']


In [23]:
from pathlib import Path
import pandas as pd
import numpy as np
from DeepTCR.DeepTCR import DeepTCR_U

REPO = Path("/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master")
DATA_FIXED = REPO / "Data/Human_HLA_Tutorial/Data_fixed"
ALL_DIR    = DATA_FIXED / "All"
HLA_CSV    = REPO / "Data/Human_HLA_Tutorial/HLA_matched.csv"

# quick sanity: show the first few columns of one file so we align indices
peek = pd.read_csv(next(ALL_DIR.glob("*.tsv")), sep="\t").head()
print("Peek columns:", list(peek.columns)[:10])
display(peek.head(3))

print("\nLoading via DeepTCR with 5-column indices...")
DTCRU = DeepTCR_U("HLA Tutorial (All, 5-col)")
DTCRU.Get_Data(
    directory=str(DATA_FIXED),
    classes=['All'],            # important: data is under Data_fixed/All
    Load_Prev_Data=False,
    sep='\t',                   # TSV files
    aa_column_beta=0,           # aminoAcid
    count_column=1,             # count
    v_beta_column=2,            # vGeneName
    d_beta_column=3,            # dGeneName
    j_beta_column=4,            # jGeneName
    hla=str(HLA_CSV),           # matched HLA file (File column = Sample_X.tsv)
    use_hla_supertype=False,
    keep_non_supertype_alleles=True,
)

# summary
n_seq = getattr(DTCRU, "num_seq", None)
if n_seq is None and hasattr(DTCRU, "sample_id"):
    n_seq = len(DTCRU.sample_id)

print("\nLoaded summary")
print("  rows (sequences):   ", n_seq)
if hasattr(DTCRU, "sample_id"):
    print("  samples (unique):   ", len(np.unique(DTCRU.sample_id)))
if hasattr(DTCRU, "lb"):
    try:
        print("  label classes:      ", list(DTCRU.lb.classes_))
    except Exception:
        pass


Peek columns: ['nucleotide', 'aminoAcid', 'count (templates/reads)', 'frequencyCount (%)', 'cdr3Length', 'vMaxResolved', 'vFamilyName', 'vGeneName', 'vGeneAllele', 'vFamilyTies']


,nucleotide,aminoAcid,count (templates/reads),frequencyCount (%),cdr3Length,vMaxResolved,vFamilyName,vGeneName,vGeneAllele,vFamilyTies,...,jOrphon,vFunction,dFunction,jFunction,fractionNucleated,vAlignLength,vAlignSubstitutionCount,vAlignSubstitutionIndexes,vAlignSubstitutionGeneThreePrimeIndexes,vSeqWithMutations
0,CAAAAGAACCCGACAGCTTTCTATCTCTGTGCCAGTAGTATTTCCG...,CASSISVASGQYF,7,0.034682,39,TCRBV19-01,TCRBV19,TCRBV19-01,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ATCCTGAAGACAGCAGCTTCTACATCTGCAGTGCTAGCGGACGGGC...,CSASGRASFYEQFF,5,0.022434,42,TCRBV20,TCRBV20,Unknown,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GACTCGGCCGTGTATGTCTGTGCCAGCAGCTTTTTAGGACAGTCCA...,CASSFLGQSTGELFF,4,0.020235,45,TCRBV11-03*01,TCRBV11,TCRBV11-03,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Loading via DeepTCR with 5-column indices...
Loading Data...


ValueError: invalid literal for int() with base 10: 'CAISTGDSNQPQHF'

In [24]:
from pathlib import Path
import pandas as pd, numpy as np
from DeepTCR.DeepTCR import DeepTCR_U

REPO = Path("/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master")
DATA_DIR = REPO / "Data/Human_HLA_Tutorial/Data_fixed" / "All"
HLA_CSV  = REPO / "Data/Human_HLA_Tutorial/HLA_matched.csv"

# Sanity check the columns we’re about to use
sample_path = sorted(DATA_DIR.glob("*.tsv"))[0]
df_head = pd.read_csv(sample_path, sep="\t").iloc[:3, [1,2,7,14,21]]
df_head.columns = ["AA (col 1)", "COUNT (col 2)", "V (col 7)", "D (col 14)", "J (col 21)"]
print("Preview of the columns DeepTCR will use:")
display(df_head)

print("\nLoading via DeepTCR (MiXCR 52-col indices)...")
DTCRU = DeepTCR_U("HLA Tutorial (MiXCR)")

DTCRU.Get_Data(
    directory=str(REPO / "Data/Human_HLA_Tutorial/Data_fixed"),
    classes=['All'],
    Load_Prev_Data=False,
    sep='\t',
    aa_column_beta=1,
    count_column=2,
    v_beta_column=7,
    d_beta_column=14,
    j_beta_column=21,
    hla=str(HLA_CSV),
    use_hla_supertype=False,
    keep_non_supertype_alleles=True,
)

# Summarize
print("\nLoaded summary")
n_samples = len(getattr(DTCRU, "sample_id", []))
print("  samples (rows):     ", n_samples)
print("  samples (unique):   ", len(np.unique(DTCRU.sample_id)) if n_samples else 0)
try:
    print("  label classes:      ", list(DTCRU.lb.classes_))
except Exception:
    pass


Preview of the columns DeepTCR will use:


,AA (col 1),COUNT (col 2),V (col 7),D (col 14),J (col 21)
0,CAISTGDSNQPQHF,7,TCRBV06-08,TCRBD01-01,TCRBJ01-05
1,CASSLFQLSNTGELFF,4,TCRBV28-01,TCRBD01-01,TCRBJ02-02
2,CSARGGPDTQYF,3,TCRBV29-01,TCRBD02-01,TCRBJ02-03



Loading via DeepTCR (MiXCR 52-col indices)...
Loading Data...


/Users/ali/Downloads/Maj_Papers/New TCR/OldDeepTCR/DeepTCR-master/.venv/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Embedding Sequences...
Data Loaded

Loaded summary
  samples (rows):      0
  samples (unique):    0
  label classes:       ['All']
